# Marketing Analytics — Customer Segmentation

**Dataset:** [Mall Customer Segmentation — Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

---

## Objective

This project applies K-Means clustering to a mall customer dataset to identify distinct groups based on income level and spending behavior. Rather than treating all customers the same, the goal is to surface segments that reflect how different types of shoppers actually engage with a retail environment — enabling more targeted marketing strategies for each group. The analysis works through the full modeling pipeline: exploratory analysis, outlier handling, feature scaling, cluster selection via the elbow method, and segment interpretation.

## Data

The dataset contains records for 200 mall customers sourced from Kaggle. Each entry captures a unique customer ID, the customer's gender and age, their annual income measured in thousands of dollars, and a spending score on a scale of 1 to 100. The spending score is assigned by the mall based on purchase frequency and transaction history, making it a composite behavioral indicator rather than a raw sales figure. This combination of demographic and behavioral features makes the dataset well-suited for segmentation work.

In [ ]:
#Install needed Libraries
import numpy as np
from scipy.stats import iqr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
!pip install plotly
import plotly.express as px

In [ ]:
#Load the data
customer_df = pd.read_csv('Mall_Customers.csv')
customer_df.head()

We will start with the first five rows of our dataset using the **head()** function and later use the **describe()** function to output a statistical summary of it.

#Check for missing data

We will need to check for any missing data.

In [ ]:
customer_df.isnull().sum()

#Describe the data

We now use the **describe()** function to output a statistical summary of our data.

In [ ]:
customer_df.describe()

In [ ]:
customer_df.info()

In [ ]:
customer_df.dtypes

In [ ]:
customer_df.shape

#Outliers

In [ ]:
list1 = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
for i in list1:
    print(str(i)+': ')
    ax = sns.boxplot(x=customer_df[str(i)])
    plt.show()

#Handling Outliers

Next we will need to remove the outliers from our data.

In [ ]:
max_age = customer_df['Age'].quantile(0.99)
max_age

In [ ]:
max_annual_income_k = customer_df['Annual Income (k$)'].quantile(0.99)
max_annual_income_k

In [ ]:
max_spending_score = customer_df['Spending Score (1-100)'].quantile(0.99)
print(max_spending_score)

# Data Analysis

##Univariate analysis


Univariate analysis entails evaluating a single feature to gain insights about it. So, the initial step in performing EDA is to undertake univariate analysis, which includes evaluating descriptive or summary statistics about the feature.

For example, check a feature distribution, the proportion of a feature, and so on.

In our case, we will check the dataset's distribution of customer's ages. We can do that by typing the following:

In [ ]:
sns.histplot(customer_df, x="Age", bins = list(range(10, 150, 10)))
plt.title("Distribution of Customer's Age")

We can see from the above summary that most of the customers belong in the age range of 30 - 40.



##Bivariate Analysis

We performed a bivariate analysis, which usually involves the correlation of two attributes at the same time.

In our case, some of the bivariate analysis we'll perform in the project include observing the average total spent across different client age groups, determining a correlation between customer income and total spending, and so on, as shown below.

In [ ]:
fig = px.scatter(customer_df, x="Annual Income (k$)",
                 y="Spending Score (1-100)",
                 title="Relationship Between Customer's Income and Spending",
                height=500,
                color_discrete_sequence = px.colors.qualitative.G10[1:])
fig.show()

##Multivariate Analysis

After we've completed our univariate (analysis of single feature) and bivariate (analysis of two features) analysis, the last phase of our EDA is to perform a Multivariate Analysis.

This consists of understanding the relationship between two or more variables.

In our project, one of the multivariate analysis we'll do is to understand the relationship between Income, Spending Score, and Gender.

In [ ]:
fig = px.scatter(
    data_frame=customer_df,
x = "Annual Income (k$)",
    y = "Spending Score (1-100)",
    facet_col = "Gender",
    title = "Relationship between Income VS Total Amount Spent Based on Age",
    color = "Age",
    height=500
)
fig.show()

We can see from the analysis that Male & Female customers between the ages of 60 to 70 years old generally spend less than other customers between the ages of 20 to 40.

This is because younger adults typically have higher disposable incomes compared to retirees. They are often in the early stages of their careers, earning regular salaries, and have fewer financial obligations such as mortgages, healthcare costs, or supporting dependents, which affects their spending habits.



#Segmentation Model

##The Elbow Method

The elbow method is the strategy we'll use to select the best cluster. It works very well by plotting the error from each cluster and looking for a spot that forms an elbow on the plot.

As a result, the ideal cluster is the one that produces that elbow.

In [ ]:
customer_df["Annual Income (k$)"].fillna(customer_df["Annual Income (k$)"].median(), inplace=True)

In [ ]:
data = customer_df[["Annual Income (k$)", "Spending Score (1-100)"]]

In [ ]:
df_log = np.log(data)

In [ ]:
std_scaler = StandardScaler()
df_scaled = std_scaler.fit_transform(df_log)

In [ ]:
errors = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(df_scaled)
    errors.append(model.inertia_)


plt.title('The Elbow Method')
plt.xlabel('k'); plt.ylabel('Error of Cluster')
sns.pointplot(x=list(range(1, 11)), y=errors)
plt.show()

Let's summarize what the above code does. We specified the number of clusters to experiment with in the **range(1, 11)**. Then, we fit the features on those clusters and added the error to the list we created above.

Following that, we plot the error for each cluster. The diagram shows that the cluster that creates the elbow is three. So, three clusters is the best value for our model. As a result, we will build the **KMeans** model utilizing three clusters.

In [ ]:
model = KMeans(n_clusters = 3, random_state=42)
model.fit(df_scaled)

In [ ]:
data = data.assign(ClusterLabel = model.labels_)

Now we've built our model. The next thing will be to assign the cluster label for each observation. So we will assign the label to the original feature we didn't processed. That is, where we assigned Annual Income and Spending Score  to the variable data

Now that we've built the model, the next step is to interpret the results from each cluster.

Depending on your goals, there are numerous ways to summarize your cluster's results. The most common summary uses central tendency, which includes mean, median, and mode.

For our case, we will use the median. We're using the median because the original features have outliers, and the mean is very sensitive to outliers.

Our next step involves aggregating the cluster labels and finding the median for Annual Income and Spending Score. We will achieve this using the Pandas groupby method. This method allows us to group the data by the cluster labels and then calculate the median for each group. The result is a clear and concise summary of the data, facilitating our interpretation of the results.

In [ ]:
data.groupby("ClusterLabel")[["Annual Income (k$)", "Spending Score (1-100)"]].median()

We can see that there is a trend within the clusters:

* Cluster 0 translates to customers who earn less and spend less.
* Cluster 1 represent customers that earn more and spend more.
* Cluster 2 represents customers that earn moderate and spend moderate.

In [ ]:
fig = px.scatter(
    data_frame=data,
    x = "Spending Score (1-100)",
    y= "Annual Income (k$)",
    title = "Relationship between Income VS Total Amount Spent",
    color = "ClusterLabel",
    height=500
)
fig.show()

In [ ]:
data = customer_df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]]
df_log = np.log(data)
std_scaler = StandardScaler()
df_scaled = std_scaler.fit_transform(df_log)

In [ ]:
model = KMeans(n_clusters=3, random_state=42)
model.fit(df_scaled)

In [ ]:
customer_df = data.assign(ClusterLabel= model.labels_)

In [ ]:
customer_df.groupby("ClusterLabel").agg({"Age":"mean", "Annual Income (k$)":"median", "Spending Score (1-100)":"median"}).round()

We can see from the above summary that:

* Cluster 0 depicts mature customers that earn a lot and also spend a lot.
* Cluster 1 translates to older customers that earn a lot and also spend a lot.
* Cluster 2 depicts young customers that earn less and also spend less.

#Visualization

We can also visualize our results by the following chart:

In [ ]:
fig = px.scatter_3d(customer_df, x="Annual Income (k$)",
                    y="Spending Score (1-100)", z="Age", color="ClusterLabel", height=550,
                   title = "Visualizing Cluster Result Using 3 Features")
fig.show()